# Contexte
Une entreprise exploite plusieurs bâtiments intelligents équipés de capteurs IoT. Chaque capteur
collecte régulièrement des informations sur la température, l'humidité, la qualité de l'air, la
consommation énergétique, le nombre de personnes présentes, le type de bâtiment, le mode de
fonctionnement et l'état du système de climatisation.
Les données collectées sont destinées à alimenter ultérieurement un modèle de Machine Learning
capable de prédire la consommation énergétique ou de détecter les situations anormales.
Cependant, les données brutes présentent volontairement différents problèmes : valeurs
manquantes, doublons, valeurs aberrantes, types incorrects, valeurs incohérentes, variables
catégorielles, catégories rares, déséquilibre des classes et échelles différentes entre variables.
L'objectif de l'atelier est donc de transformer le fichier brut en un jeu de données propre et prêt
pour le Machine Learning.

# Importation des bibliotheques  

In [90]:

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

# Partie 1 – Explorer les données

## 1) Charger les données CSV

In [91]:
df = pd.read_csv("/home/adama/atelier_prepa_donnees_tab/data/smart_building_raw.csv")
brut = df.copy()          # on garde une copie du brut pour comparer à la fin

## 2) Afficher les premières lignes du dataset

In [92]:
df.head()

,id_mesure,date,batiment,type_batiment,zone,temperature,humidite,co2,occupation,consommation_kwh,mode_climatisation,etat_systeme,jour_semaine,alerte
0,1174,2025-02-13 06:00:00,B8,Entrepôt,A,23.0,40.0,851.0,26.0,138.4,Eco,Normal,Jeudi,Non
1,1275,2025-03-10 12:00:00,B7,Bureau,D,28.0,69.0,538.0,0.0,170.2,Eco,Normal,Lundi,Non
2,1493,2025-05-04 00:00:00,B5,Centre commercial,B,19.5,52.3,1198.0,64.0,149.8,Normal,Normal,Dimanche,Oui
3,1073,2025-01-19 00:00:00,B6,Université,D,20.0,58.4,1014.0,30.0,141.6,Normal,Normal,Dimanche,Non
4,1454,2025-04-24 06:00:00,B7,Bureau,D,26.8,70.5,628.0,42.0,228.3,Normal,Normal,Jeudi,Non


## 3) Afficher les dernières lignes du dataset

In [93]:
df.tail()

,id_mesure,date,batiment,type_batiment,zone,temperature,humidite,co2,occupation,consommation_kwh,mode_climatisation,etat_systeme,jour_semaine,alerte
502,1107,2025-01-27 12:00:00,B2,École,C,29.8,66.5,778.0,48.0,243.5,Normal,Normal,Lundi,Non
503,1271,2025-03-09 12:00:00,B3,Hôpital,B,21.1,66.3,3900.0,53.0,145.8,Eco,Alerte,Dimanche,Oui
504,1349,2025-03-29 00:00:00,B5,Centre commercial,B,21.6,61.9,959.0,33.0,100.2,Eco,Normal,Samedi,Non
505,1436,2025-04-19 18:00:00,B7,Bureau,C,21.2,54.5,1066.0,38.0,150.9,Boost,Normal,Samedi,Oui
506,1103,2025-01-26 12:00:00,B2,École,D,NaN,64.8,584.0,29.0,163.4,Normal,Panne,Dimanche,Non


## 4) Combien d'observations contient le dataset ?

In [94]:
df.shape[0]

507

## 5) Combien de variables possède le dataset ?

In [95]:
df.shape[1]

14

## 6) Identifier les variables numériques ;

In [96]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 507 entries, 0 to 506
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_mesure           507 non-null    int64  
 1   date                507 non-null    str    
 2   batiment            507 non-null    str    
 3   type_batiment       503 non-null    str    
 4   zone                507 non-null    str    
 5   temperature         495 non-null    float64
 6   humidite            496 non-null    float64
 7   co2                 500 non-null    float64
 8   occupation          501 non-null    float64
 9   consommation_kwh    502 non-null    float64
 10  mode_climatisation  502 non-null    str    
 11  etat_systeme        507 non-null    str    
 12  jour_semaine        502 non-null    str    
 13  alerte              507 non-null    str    
dtypes: float64(5), int64(1), str(8)
memory usage: 55.6 KB


In [97]:
variables_numerique = df[["temperature", "humidite", "co2", "occupation", "consommation_kwh"]]
variables_numerique

,temperature,humidite,co2,occupation,consommation_kwh
0,23.0,40.0,851.0,26.0,138.4
1,28.0,69.0,538.0,0.0,170.2
2,19.5,52.3,1198.0,64.0,149.8
3,20.0,58.4,1014.0,30.0,141.6
4,26.8,70.5,628.0,42.0,228.3
...,...,...,...,...,...
502,29.8,66.5,778.0,48.0,243.5
503,21.1,66.3,3900.0,53.0,145.8
504,21.6,61.9,959.0,33.0,100.2
505,21.2,54.5,1066.0,38.0,150.9


## 7) Identifier les variables catégorielles

In [98]:
variables_categorique = df[["batiment", "type_batiment", "zone", "mode_climatisation", "etat_systeme", "jour_semaine", "alerte" ]]
variables_categorique

,batiment,type_batiment,zone,mode_climatisation,etat_systeme,jour_semaine,alerte
0,B8,Entrepôt,A,Eco,Normal,Jeudi,Non
1,B7,Bureau,D,Eco,Normal,Lundi,Non
2,B5,Centre commercial,B,Normal,Normal,Dimanche,Oui
3,B6,Université,D,Normal,Normal,Dimanche,Non
4,B7,Bureau,D,Normal,Normal,Jeudi,Non
...,...,...,...,...,...,...,...
502,B2,École,C,Normal,Normal,Lundi,Non
503,B3,Hôpital,B,Eco,Alerte,Dimanche,Oui
504,B5,Centre commercial,B,Eco,Normal,Samedi,Non
505,B7,Bureau,C,Boost,Normal,Samedi,Oui


## 8) Identifier les dates

In [99]:
df["date"] = pd.to_datetime(df["date"], errors="coerce")
Variables_date = df[["date"]]
Variables_date

,date
0,2025-02-13 06:00:00
1,2025-03-10 12:00:00
2,2025-05-04 00:00:00
3,2025-01-19 00:00:00
4,2025-04-24 06:00:00
...,...
502,2025-01-27 12:00:00
503,2025-03-09 12:00:00
504,2025-03-29 00:00:00
505,2025-04-19 18:00:00


## 9) Identifier les identifiants

In [100]:
variables_identifiant = df[["id_mesure"]]
variables_identifiant

,id_mesure
0,1174
1,1275
2,1493
3,1073
4,1454
...,...
502,1107
503,1271
504,1349
505,1436


## 10) Déterminer les statistiques : moyenne, médiane, minimum, maximum, écart-type et quartiles.

In [101]:
variables_numerique.describe().T

,count,mean,std,min,25%,50%,75%,max
temperature,495.0,24.154141,7.418465,-30.0,21.600,24.00,26.000,96.0
humidite,496.0,57.864113,16.026336,-12.0,49.275,57.55,65.750,160.0
co2,500.0,844.150000,582.181386,89.0,623.750,787.50,952.000,6000.0
occupation,501.0,44.850299,24.949139,-20.0,27.000,46.00,61.000,116.0
consommation_kwh,502.0,169.069323,53.164294,-100.0,136.875,169.80,202.975,336.2


## 11) Y a-t-il des variables potentiellement problématiques ?

D’après les statistiques présentées ci-dessus, plusieurs variables semblent présenter des valeurs potentiellement problématiques :

Température : les valeurs minimales et maximales observées sont très extrêmes et peuvent être physiquement peu plausibles dans le contexte d’un bâtiment.

Humidité : cette variable devrait normalement être comprise entre 0 % et 100 %. Cependant, certaines valeurs observées se situent en dehors de cet intervalle.

CO₂ : certaines valeurs apparaissent particulièrement élevées par rapport à l’ensemble de la distribution, ce qui peut indiquer la présence de valeurs aberrantes.

Occupation : cette variable représentant le nombre de personnes présentes dans le bâtiment, elle ne devrait pas prendre de valeurs négatives.

Consommation_kWh : une consommation énergétique ne peut normalement pas être négative. Les valeurs négatives doivent donc être vérifiées.

Type_bâtiment et mode_climatisation : ces variables catégorielles peuvent contenir des incohérences au niveau des catégories, notamment des différences d’écriture, des espaces superflus ou des variations de casse.

## 12) Pour les données incohérentes :

### a) rechercher des valeurs telles que humidité < 0 ;

In [102]:
var_humi_inferieure_0 = df[df["humidite"] < 0]
print("Nombre de lignes avec humidité inférieure à 0 :", var_humi_inferieure_0.shape[0], "sur un total de", df.shape[0], "lignes.")
var_humi_inferieure_0[["id_mesure", "humidite"]]

Nombre de lignes avec humidité inférieure à 0 : 3 sur un total de 507 lignes.


,id_mesure,humidite
281,1126,-5.0
335,1036,-8.0
366,1216,-12.0
